# Hourly LSTM training — agent-final

Trains **Single / Double / BiLSTM** on clean preprocessed data (no CO2 leakage).

**Before this:** run `preprocess.ipynb` → saves `outputs/preprocess/hourly/data.csv`

**No SHAP here** — XAI comes later.

Run cells top to bottom in Colab.

In [1]:
window_sizes = [1, 4, 8, 12, 16, 24, 36, 48, 74, 168, 336, 672]
epochs = 50
batch_size = 32
test_ratio = 0.18


from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Shared-Colab-Storage/agent-final"

DATA_CSV = f"{BASE}/outputs/preprocess/hourly/data.csv"
SCALER_PKL = f"{BASE}/outputs/preprocess/hourly/scaler.pkl"
SAVE_PATH = f"{BASE}/outputs/train/hourly"

print("DATA:", DATA_CSV)
print("SAVE:", SAVE_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA: /content/drive/MyDrive/Shared-Colab-Storage/agent-final/outputs/preprocess/hourly/data.csv
SAVE: /content/drive/MyDrive/Shared-Colab-Storage/agent-final/outputs/train/hourly


In [2]:
!pip install -q pandas numpy scikit-learn matplotlib tensorflow

import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.models import Sequential, load_model

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_true - y_pred)))

os.makedirs(SAVE_PATH, exist_ok=True)

## Load preprocessed hourly data

In [3]:
scaled_df = pd.read_csv(DATA_CSV)
with open(SCALER_PKL, "rb") as f:
    scaler = pickle.load(f)

feature_cols = list(scaled_df.columns)
target_idx = 0  
n_features = len(feature_cols)

print("Rows:", len(scaled_df))
print("Features (9 after preprocess):", feature_cols)
scaled_df.head()

Rows: 8760
Features (9 after preprocess): ['Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor', 'WeekStatus', 'Day_of_week', 'Load_Type', 'hour_sin', 'hour_cos']


,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Lagging_Current_Power_Factor,Leading_Current_Power_Factor,WeekStatus,Day_of_week,Load_Type,hour_sin,hour_cos
0,0.007306,0.049021,0.497919,1.0,0.0,0.166667,0.0,0.549246,1.000000
1,0.007638,0.054599,0.437513,1.0,0.0,0.166667,0.0,0.676977,0.970217
2,0.007840,0.057752,0.407121,1.0,0.0,0.166667,0.0,0.792648,0.908390
3,0.007288,0.056123,0.412628,1.0,0.0,0.166667,0.0,0.888375,0.818731
4,0.008484,0.061112,0.386565,1.0,0.0,0.166667,0.0,0.957636,0.707352


## Sliding windows

In [4]:
arr = scaled_df[feature_cols].to_numpy(dtype=np.float32)

data = {}
for window in window_sizes:
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i + window])
        y.append(arr[i + window, target_idx])
    X, y = np.array(X), np.array(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_ratio, shuffle=False
    )
    data[f"win{window}"] = dict(
        X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test
    )
    print(f"win{window}: train {X_train.shape}, test {X_test.shape}")

win1: train (7182, 1, 9), test (1577, 1, 9)
win4: train (7179, 4, 9), test (1577, 4, 9)
win8: train (7176, 8, 9), test (1576, 8, 9)
win12: train (7173, 12, 9), test (1575, 12, 9)
win16: train (7170, 16, 9), test (1574, 16, 9)
win24: train (7163, 24, 9), test (1573, 24, 9)
win36: train (7153, 36, 9), test (1571, 36, 9)
win48: train (7143, 48, 9), test (1569, 48, 9)
win74: train (7122, 74, 9), test (1564, 74, 9)
win168: train (7045, 168, 9), test (1547, 168, 9)
win336: train (6907, 336, 9), test (1517, 336, 9)
win672: train (6632, 672, 9), test (1456, 672, 9)


## Train models (single / double / bidir)

Skips training if `winN.keras` already exists (set `FORCE_RETRAIN=True` to redo).

In [5]:
stacks = ["single", "double", "bidir"]
trained = {}

for stack in stacks:
    print(f"\n========== {stack.upper()} LSTM ==========")
    model_dir = os.path.join(SAVE_PATH, stack, "models")
    hist_dir = os.path.join(SAVE_PATH, stack, "history")
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(hist_dir, exist_ok=True)
    trained[stack] = {}

    for window in window_sizes:
        model_path = os.path.join(model_dir, f"win{window}.keras")
        hist_path = os.path.join(hist_dir, f"win{window}.pkl")
        d = data[f"win{window}"]

        if os.path.exists(model_path):
            model = load_model(model_path, custom_objects={"rmse": rmse})
            hist = pickle.load(open(hist_path, "rb")) if os.path.exists(hist_path) else {}
            print(f"  win{window}: loaded")
            trained[stack][window] = {"model": model, "history": hist}
            continue

        print(f"  win{window}: training...")
        model = Sequential()
        if stack == "single":
            model.add(LSTM(64, input_shape=(window, n_features)))
        elif stack == "double":
            model.add(LSTM(64, return_sequences=True, input_shape=(window, n_features)))
            model.add(LSTM(64))
        else:
            model.add(Bidirectional(LSTM(64), input_shape=(window, n_features)))
        model.add(Dropout(0.1))
        model.add(Dense(1))
        model.compile(optimizer="adam", loss=rmse, metrics=["mae"])

        hist = model.fit(
            d["X_train"], d["y_train"],
            validation_data=(d["X_test"], d["y_test"]),
            epochs=epochs,
            batch_size=batch_size,
            verbose=1,
        )
        model.save(model_path)
        pickle.dump(hist.history, open(hist_path, "wb"))
        trained[stack][window] = {"model": model, "history": hist.history}
        print(f"  win{window}: saved")


========== SINGLE LSTM ==========
  win1: training...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.1455 - mae: 0.1067 - val_loss: 0.1181 - val_mae: 0.0843
Epoch 2/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1263 - mae: 0.0851 - val_loss: 0.1100 - val_mae: 0.0754
Epoch 3/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1250 - mae: 0.0833 - val_loss: 0.1137 - val_mae: 0.0788
Epoch 4/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1249 - mae: 0.0827 - val_loss: 0.1090 - val_mae: 0.0740
Epoch 5/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1231 - mae: 0.0817 - val_loss: 0.1102 - val_mae: 0.0735
Epoch 6/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1221 - mae: 0.0815 - val_loss: 0.1100 - val_mae: 0.0730
Epoch 7/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1219 - mae: 0.0808 - val_loss: 0.1092 - val_mae: 0.0727
Epoch 8/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1218 - mae: 0.0814 - val_loss: 0.1072 - val_mae: 0.0722
Epoch 9/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - lo

KeyboardInterrupt: 

## 4) Evaluate (RMSE, MAE, R², WIA in kWh)

In [ ]:
def to_kwh(values):
    values = np.asarray(values).ravel()
    d = np.zeros((len(values), n_features))
    d[:, target_idx] = values
    return scaler.inverse_transform(d)[:, target_idx]

rows = []
for stack in stacks:
    for window in window_sizes:
        if window not in trained[stack]:
            continue
        model = trained[stack][window]["model"]
        y_test = data[f"win{window}"]["y_test"]
        y_pred = model.predict(data[f"win{window}"]["X_test"], verbose=0).ravel()

        yt, yp = to_kwh(y_test), to_kwh(y_pred)
        rmse_k = float(np.sqrt(mean_squared_error(yt, yp)))
        mae_k = float(mean_absolute_error(yt, yp))
        r2_k = float(r2_score(yt, yp))
        mean_obs = np.mean(yt)
        wia_k = float(1 - np.sum((yp - yt) ** 2) / np.sum((np.abs(yp - mean_obs) + np.abs(yt - mean_obs)) ** 2))

        rows.append({
            "model": stack, "window": window,
            "rmse_kwh": rmse_k, "mae_kwh": mae_k, "r2": r2_k, "wia": wia_k,
        })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(os.path.join(SAVE_PATH, "results_metrics.csv"), index=False)

best = metrics_df.loc[metrics_df["rmse_kwh"].idxmin()]
print("Best model:")
print(best)
metrics_df.sort_values("rmse_kwh").head(10)

## 5) RMSE heatmap (quick view)

In [ ]:
pivot = metrics_df.pivot(index="window", columns="model", values="rmse_kwh")
plt.figure(figsize=(6, 8))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd_r")
plt.title("Hourly — RMSE (kWh) by window & architecture")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "rmse_heatmap.png"), dpi=120)
plt.show()

## Done

Saved under `outputs/train/hourly/`:
- `{single,double,bidir}/models/winN.keras`
- `results_metrics.csv`
- `rmse_heatmap.png`

Next: XAI / behavior analysis notebook (later).